## 1️⃣ Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Mount Google Drive for saving models
from google.colab import drive
drive.mount('/content/drive')

# Create project directory in Drive
!mkdir -p /content/drive/MyDrive/CyberbullyingDetection/models

In [ ]:
# Clone the repository
!rm -rf /content/Cyberbullying-detection
!git clone https://github.com/vinod-hn/Cyberbullying-detection.git
%cd /content/Cyberbullying-detection
!git lfs pull  # Pull large model files if needed

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate
!pip install -q scikit-learn pandas numpy matplotlib seaborn
!pip install -q torch torchvision torchaudio --upgrade
!pip install -q emoji indic-transliteration
!pip install -q sentencepiece protobuf

print("\n✅ All dependencies installed!")

## 2️⃣ Load and Explore Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add project root to path
PROJECT_ROOT = '/content/Cyberbullying-detection'
sys.path.insert(0, PROJECT_ROOT)

# Load datasets
DATA_PATH = os.path.join(PROJECT_ROOT, '00_data', 'processed')

train_df = pd.read_csv(os.path.join(DATA_PATH, 'train_data.csv'))
val_df = pd.read_csv(os.path.join(DATA_PATH, 'val_data.csv'))
test_df = pd.read_csv(os.path.join(DATA_PATH, 'test_data.csv'))

print(f"📊 Dataset Sizes:")
print(f"   Train: {len(train_df):,} samples")
print(f"   Val:   {len(val_df):,} samples")
print(f"   Test:  {len(test_df):,} samples")
print(f"\n📋 Columns: {list(train_df.columns)}")
print(f"\n🔍 Sample data:")
train_df.head()

In [ ]:
# Visualize label distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, df) in zip(axes, [('Train', train_df), ('Validation', val_df), ('Test', test_df)]):
    label_col = 'label' if 'label' in df.columns else df.columns[-1]
    df[label_col].value_counts().plot(kind='bar', ax=ax, color=['green', 'red'])
    ax.set_title(f'{name} Label Distribution')
    ax.set_xlabel('Label')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3️⃣ Preprocessing

In [ ]:
# Import preprocessing modules
from importlib import import_module

try:
    from preprocessing import text_normalizer, emoji_handler, slang_expander
    print("✅ Preprocessing modules loaded from package")
except ImportError:
    # Manual import if package structure differs
    sys.path.insert(0, os.path.join(PROJECT_ROOT, '01_preprocessing'))
    import text_normalizer
    import emoji_handler
    import slang_expander
    print("✅ Preprocessing modules loaded directly")

In [ ]:
# Simple preprocessing function
import re
import emoji

def preprocess_text(text):
    """Clean and normalize text for model input."""
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    
    # Convert emojis to text
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text.strip()

# Determine text column
text_col = 'text' if 'text' in train_df.columns else train_df.columns[0]
label_col = 'label' if 'label' in train_df.columns else train_df.columns[-1]

print(f"📝 Text column: {text_col}")
print(f"🏷️ Label column: {label_col}")

# Apply preprocessing
train_df['processed_text'] = train_df[text_col].apply(preprocess_text)
val_df['processed_text'] = val_df[text_col].apply(preprocess_text)
test_df['processed_text'] = test_df[text_col].apply(preprocess_text)

print(f"\n✅ Preprocessing complete!")
print(f"\n🔍 Sample processed text:")
print(train_df[['processed_text', label_col]].head())

## 4️⃣ Baseline Models (TF-IDF + SVM/Naive Bayes)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import pickle

# Encode labels
le = LabelEncoder()
y_train = le.fit_transform(train_df[label_col])
y_val = le.transform(val_df[label_col])
y_test = le.transform(test_df[label_col])

print(f"📋 Classes: {le.classes_}")

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_train_tfidf = tfidf.fit_transform(train_df['processed_text'])
X_val_tfidf = tfidf.transform(val_df['processed_text'])
X_test_tfidf = tfidf.transform(test_df['processed_text'])

print(f"✅ TF-IDF features: {X_train_tfidf.shape[1]:,}")

In [ ]:
# Train baseline models
baseline_models = {
    'Naive Bayes': MultinomialNB(),
    'SVM': SVC(kernel='linear', probability=True),
    'Logistic Regression': LogisticRegression(max_iter=1000)
}

baseline_results = {}

for name, model in baseline_models.items():
    print(f"\n🔄 Training {name}...")
    model.fit(X_train_tfidf, y_train)
    
    # Evaluate on validation set
    y_pred = model.predict(X_val_tfidf)
    acc = accuracy_score(y_val, y_pred)
    baseline_results[name] = {'model': model, 'accuracy': acc}
    
    print(f"   ✅ {name} Validation Accuracy: {acc:.4f}")
    print(classification_report(y_val, y_pred, target_names=le.classes_))

In [ ]:
# Save best baseline model
best_baseline = max(baseline_results.items(), key=lambda x: x[1]['accuracy'])
print(f"\n🏆 Best Baseline: {best_baseline[0]} (Accuracy: {best_baseline[1]['accuracy']:.4f})")

# Save to Drive
SAVE_PATH = '/content/drive/MyDrive/CyberbullyingDetection/models'

with open(f'{SAVE_PATH}/best_baseline_model.pkl', 'wb') as f:
    pickle.dump(best_baseline[1]['model'], f)

with open(f'{SAVE_PATH}/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open(f'{SAVE_PATH}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print(f"✅ Baseline model saved to Google Drive!")

## 5️⃣ Transformer Model (BERT/mBERT) - GPU Training

In [ ]:
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
import torch

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

# Choose model - mBERT for multilingual (Kannada + English)
MODEL_NAME = 'bert-base-multilingual-cased'  # or 'ai4bharat/indic-bert' for Indic languages

print(f"📦 Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=len(le.classes_)
).to(device)

print(f"✅ Model loaded! Parameters: {model.num_parameters():,}")

In [ ]:
# Prepare datasets for Transformers
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

# Create Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df[['processed_text', label_col]].rename(columns={'processed_text': 'text', label_col: 'label'}))
val_dataset = Dataset.from_pandas(val_df[['processed_text', label_col]].rename(columns={'processed_text': 'text', label_col: 'label'}))
test_dataset = Dataset.from_pandas(test_df[['processed_text', label_col]].rename(columns={'processed_text': 'text', label_col: 'label'}))

# Encode labels if string
def encode_labels(example):
    example['label'] = le.transform([example['label']])[0]
    return example

if train_dataset['label'][0].__class__ == str:
    train_dataset = train_dataset.map(encode_labels)
    val_dataset = val_dataset.map(encode_labels)
    test_dataset = test_dataset.map(encode_labels)

# Tokenize
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print(f"✅ Datasets prepared!")
print(f"   Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

In [ ]:
# Define metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Training arguments optimized for Colab GPU
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,  # Adjust based on GPU memory
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=True,  # Mixed precision for faster training
    gradient_accumulation_steps=2,
    dataloader_num_workers=2,
    report_to='none'
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("✅ Trainer configured! Ready for training.")

In [ ]:
# Train the model!
print("🚀 Starting training...\n")
train_result = trainer.train()

print(f"\n✅ Training complete!")
print(f"   Total steps: {train_result.global_step}")
print(f"   Training loss: {train_result.training_loss:.4f}")

In [ ]:
# Evaluate on test set
print("📊 Evaluating on test set...\n")
test_results = trainer.evaluate(test_dataset)

print("Test Results:")
for key, value in test_results.items():
    print(f"   {key}: {value:.4f}")

# Get predictions for detailed report
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)

print("\n📋 Classification Report:")
print(classification_report(test_dataset['label'], preds, target_names=le.classes_))

In [ ]:
# Save transformer model to Google Drive
TRANSFORMER_SAVE_PATH = f'{SAVE_PATH}/transformer_model'

trainer.save_model(TRANSFORMER_SAVE_PATH)
tokenizer.save_pretrained(TRANSFORMER_SAVE_PATH)

# Save test metrics
import json
with open(f'{TRANSFORMER_SAVE_PATH}/test_metrics.json', 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"✅ Transformer model saved to: {TRANSFORMER_SAVE_PATH}")

## 6️⃣ Ensemble Model

In [ ]:
# Create ensemble predictions combining baseline and transformer
from scipy.special import softmax

# Get baseline predictions (probabilities)
baseline_probs = best_baseline[1]['model'].predict_proba(X_test_tfidf)

# Get transformer predictions (logits → probabilities)
transformer_probs = softmax(predictions.predictions, axis=1)

# Weighted ensemble (adjust weights as needed)
BASELINE_WEIGHT = 0.3
TRANSFORMER_WEIGHT = 0.7

ensemble_probs = BASELINE_WEIGHT * baseline_probs + TRANSFORMER_WEIGHT * transformer_probs
ensemble_preds = ensemble_probs.argmax(axis=1)

# Evaluate ensemble
print("🔀 Ensemble Model Results:\n")
print(classification_report(y_test, ensemble_preds, target_names=le.classes_))

ensemble_acc = accuracy_score(y_test, ensemble_preds)
print(f"\n🏆 Ensemble Accuracy: {ensemble_acc:.4f}")

In [ ]:
# Save ensemble configuration
ensemble_config = {
    'baseline_model': best_baseline[0],
    'baseline_weight': BASELINE_WEIGHT,
    'transformer_model': MODEL_NAME,
    'transformer_weight': TRANSFORMER_WEIGHT,
    'ensemble_accuracy': ensemble_acc
}

with open(f'{SAVE_PATH}/ensemble_config.json', 'w') as f:
    json.dump(ensemble_config, f, indent=2)

print(f"✅ Ensemble configuration saved!")

## 7️⃣ Model Comparison & Visualization

In [ ]:
# Compare all models
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Collect results
model_results = {
    'Naive Bayes': accuracy_score(y_test, baseline_models['Naive Bayes'].predict(X_test_tfidf)),
    'SVM': accuracy_score(y_test, baseline_models['SVM'].predict(X_test_tfidf)),
    'Logistic Regression': accuracy_score(y_test, baseline_models['Logistic Regression'].predict(X_test_tfidf)),
    'Transformer (mBERT)': test_results['eval_accuracy'],
    'Ensemble': ensemble_acc
}

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
ax1 = axes[0]
models = list(model_results.keys())
accuracies = list(model_results.values())
colors = ['#3498db', '#3498db', '#3498db', '#e74c3c', '#2ecc71']
bars = ax1.barh(models, accuracies, color=colors)
ax1.set_xlabel('Accuracy')
ax1.set_title('Model Accuracy Comparison')
ax1.set_xlim(0, 1)
for bar, acc in zip(bars, accuracies):
    ax1.text(acc + 0.01, bar.get_y() + bar.get_height()/2, f'{acc:.4f}', va='center')

# Confusion matrix for best model (ensemble)
ax2 = axes[1]
cm = confusion_matrix(y_test, ensemble_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_, ax=ax2)
ax2.set_title('Ensemble Model Confusion Matrix')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Actual')

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Comparison chart saved to Google Drive!")

## 8️⃣ Inference Example

In [ ]:
def predict_cyberbullying(text):
    """Predict if text is cyberbullying using ensemble model."""
    # Preprocess
    processed = preprocess_text(text)
    
    # Baseline prediction
    tfidf_vec = tfidf.transform([processed])
    baseline_prob = best_baseline[1]['model'].predict_proba(tfidf_vec)
    
    # Transformer prediction
    inputs = tokenizer(processed, return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    transformer_prob = softmax(outputs.logits.cpu().numpy(), axis=1)
    
    # Ensemble
    ensemble_prob = BASELINE_WEIGHT * baseline_prob + TRANSFORMER_WEIGHT * transformer_prob
    prediction = le.inverse_transform([ensemble_prob.argmax()])[0]
    confidence = ensemble_prob.max()
    
    return {
        'text': text,
        'prediction': prediction,
        'confidence': float(confidence),
        'probabilities': {cls: float(p) for cls, p in zip(le.classes_, ensemble_prob[0])}
    }

# Test examples
test_texts = [
    "You're such a loser, nobody likes you!",
    "Great job on the presentation today!",
    "I hate you so much, go die!",
    "Thanks for helping me with the project 😊"
]

print("🔮 Inference Examples:\n")
for text in test_texts:
    result = predict_cyberbullying(text)
    emoji_icon = "🚨" if result['prediction'] != 'Not Cyberbullying' else "✅"
    print(f"{emoji_icon} Text: \"{text}\"")
    print(f"   Prediction: {result['prediction']} (Confidence: {result['confidence']:.2%})")
    print()

## 9️⃣ Download Models

In [ ]:
# Create zip of all models for download
import shutil

# Zip the models folder
!cd /content/drive/MyDrive/CyberbullyingDetection && zip -r /content/cyberbullying_models.zip models/

# Download
from google.colab import files
files.download('/content/cyberbullying_models.zip')

print("\n✅ Models downloaded! You can also find them in your Google Drive.")

## 📋 Summary

### Models Trained:
1. **Baseline Models** (TF-IDF):
   - Naive Bayes
   - SVM
   - Logistic Regression

2. **Transformer Model**:
   - mBERT (multilingual BERT)
   - GPU-accelerated training with mixed precision

3. **Ensemble Model**:
   - Weighted combination of baseline + transformer

### Files Saved to Google Drive:
- `models/best_baseline_model.pkl`
- `models/tfidf_vectorizer.pkl`
- `models/label_encoder.pkl`
- `models/transformer_model/` (full model + tokenizer)
- `models/ensemble_config.json`
- `models/model_comparison.png`

### Next Steps:
1. Download models and integrate into your project
2. Fine-tune hyperparameters for better performance
3. Try other transformer models (IndicBERT, XLM-RoBERTa)
4. Deploy as API using the 06_api module